In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split

In [2]:
import sys
from pathlib import Path

github_root = Path.cwd().resolve().parents[2]
sys.path.append(str(github_root))

In [3]:
DIAGNOSIS_RECORDS_TRAIN = "../data/diagnosis_records_train.csv"
PARTICIPANT_DATA_TRAIN = "../data/participant_data_train.csv"
PARTICIPANT_DATA_TEST = "../data/participant_data_test.csv"
WEARABLE_DATA_TRAIN = "../data/wearable_data_train.csv" 

In [11]:
diagnosis_records_train = pd.read_csv(DIAGNOSIS_RECORDS_TRAIN)
participant_data_train = pd.read_csv(PARTICIPANT_DATA_TRAIN)
participant_data_test = pd.read_csv(PARTICIPANT_DATA_TEST)
wearable_data_train = pd.read_csv(WEARABLE_DATA_TRAIN)

In [6]:
def split_data_by_patient(df_patient, df_wearable, df_diagnosis, test_size=0.5, random_state=42):
    """
    Splits all three datasets based on participant IDs to prevent data leakage.
    
    Parameters:
    - df_patient: Participant Health Profile (1 row per ID)
    - df_wearable: Wearable Time Series (Multiple rows per ID)
    - df_diagnosis: Diagnosis records (Multiple rows per ID)
    """
    
    # 1. Split the primary patient IDs
    # Since df_patient has one row per ID, we split the dataframe directly
    train_patient, val_patient = train_test_split(
        df_patient, 
        test_size=test_size, 
        random_state=random_state,
        stratify=df_patient['outcome']  # Recommended to maintain mortality balance
    )
    
    # 2. Extract the IDs for each split
    train_ids = train_patient['ID'].unique()
    val_ids = val_patient['ID'].unique()
    
    # 3. Filter the Wearable Dataset
    # We keep only records belonging to the IDs in the respective splits
    train_wearable = df_wearable[df_wearable['ID'].isin(train_ids)].copy()
    val_wearable = df_wearable[df_wearable['ID'].isin(val_ids)].copy()
    
    # 4. Filter the Diagnosis Dataset
    train_diagnosis = df_diagnosis[df_diagnosis['ID'].isin(train_ids)].copy()
    val_diagnosis = df_diagnosis[df_diagnosis['ID'].isin(val_ids)].copy()
    
    return (train_patient, val_patient, 
            train_wearable, val_wearable, 
            train_diagnosis, val_diagnosis)

In [19]:
class MortalityDataTransformer:
    def __init__(self):
        self.feature_cols = [
            'age', 'bmi', 'smoker', 'diagnosed_afib', 'diagnosed_diabetes', 
            'diagnosed_hypertension', 'engagement_level', 'diagnosis_count',
            'sleep_quality', 'exercise_quality', 'improving_health', 'is_female'
        ]
        self.global_means = None

    def transform(self, df_patient, df_wearable, df_diagnosis):
        """
        Transforms raw dataframes into a single feature matrix.
        """
        # 1. Process Diagnosis Data [cite: 173]
        # Group by ID and count medical events
        diag_counts = df_diagnosis.groupby('ID').size().rename('diagnosis_count')
        df = df_patient.merge(diag_counts, on='ID', how='left').fillna({'diagnosis_count': 0})

        # 2. Process Wearable Data [cite: 158, 160, 164]
        # Aggregate daily measurements to patient-level means
        # Handling missing values by taking the mean per ID
        w_agg = df_wearable.groupby('ID').agg({
            'sleep_deep': 'mean',
            'sleep_rem': 'mean',
            'exercise_duration': 'mean',
            'steps': 'mean',
            'timestamp': ['min', 'max']
        })
        w_agg.columns = ['avg_deep', 'avg_rem', 'avg_ex_dur', 'avg_steps', 'start', 'end']

        # 3. Feature Engineering: Sleep & Exercise Quality
        df = df.merge(w_agg, on='ID', how='left')
        
        # Sleep Quality: Sum of Deep and REM sleep 
        df['sleep_quality'] = (df['avg_deep'] + df['avg_rem']).fillna(0)
        
        # Exercise Quality: Interaction of duration and step count [cite: 164]
        df['exercise_quality'] = (df['avg_ex_dur'] * df['avg_steps']).fillna(0)

        # 4. Improving Health Trend [cite: 146]
        # Using engagement_level as a proxy for proactive health management
        df['improving_health'] = (df['engagement_level'] > 3).astype(int)

        # 5. Demographics & Flags [cite: 146]
        df['is_female'] = (df['sex'] == 'F').astype(int)

        # 6. Final Clean-up and Feature Selection
        # Keep only required columns
        final_df = df[['ID'] + self.feature_cols].copy()

        # Handle missing values using global means (fit during training)
        if self.global_means is None:
            self.global_means = final_df[self.feature_cols].mean()
        
        final_df[self.feature_cols] = final_df[self.feature_cols].fillna(self.global_means)

        return final_df

In [17]:
(train_p, val_p, train_w, val_w, train_d, val_d) = split_data_by_patient(participant_data_train, wearable_data_train, diagnosis_records_train)

In [36]:
transformer = MortalityDataTransformer()
# 1. Transform the data
train_features = transformer.transform(train_p, train_w, train_d)
val_features = transformer.transform(val_p, val_w, val_d)

# 2. Set ID as the index for features
# This moves 'ID' from a column to the index, leaving only numerical data for X
X_train = train_features.set_index('ID')
X_val = val_features.set_index('ID')

# 3. Align and index the outcomes
# By setting 'ID' as index here, y_train.index will match X_train.index exactly
y_train = train_p.set_index('ID').loc[X_train.index]['outcome']
y_val = val_p.set_index('ID').loc[X_val.index]['outcome']

# 4. Verify alignment (Crucial for actuarial technical credibility)
print(f"Train alignment check: {X_train.index.equals(y_train.index)}")
print(f"Val alignment check: {X_val.index.equals(y_val.index)}")
vail_outcomes = val_p.set_index('ID').loc[val_features['ID']]['outcome']

Train alignment check: True
Val alignment check: True


In [43]:
X_train_const = sm.add_constant(X_train)
advanced_model = sm.Logit(y_train, X_train_const).fit()
X_val_const = sm.add_constant(X_val, has_constant='add')

Optimization terminated successfully.
         Current function value: 0.261728
         Iterations 7


In [44]:
def evaluate_mortality_model(actual_outcomes, predicted_probs):
    """
    Evaluates predictive performance, calibration, and discrimination.
    """
    # 1. Log-Loss (Competition Primary Metric) [cite: 50, 107]
    ll_score = log_loss(actual_outcomes, predicted_probs)
    
    # 2. Brier Score (Calibration Metric)
    brier_score = brier_score_loss(actual_outcomes, predicted_probs)
    
    # 3. C-index / AUC (Discrimination Metric)
    # Measures the probability that a person who died was assigned 
    # a higher risk than a person who survived.
    c_index = roc_auc_score(actual_outcomes, predicted_probs)
    
    # 4. Actuarial Reasonableness (A/E Ratio) [cite: 89, 90]
    mean_actual = np.mean(actual_outcomes)
    mean_pred = np.mean(predicted_probs)
    ae_ratio = mean_actual / mean_pred if mean_pred != 0 else 0
    
    return {
        "Log-Loss": round(ll_score, 5),
        "Brier Score": round(brier_score, 5),
        "C-index (AUC)": round(c_index, 5),
        "Actual Mortality Rate": round(mean_actual, 5),
        "Predicted Mortality Rate": round(mean_pred, 5),
        "A/E Ratio": round(ae_ratio, 3)
    }

In [49]:
advanced_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                outcome   No. Observations:                10000
Model:                          Logit   Df Residuals:                     9987
Method:                           MLE   Df Model:                           12
Date:                Sun, 19 Apr 2026   Pseudo R-squ.:                 0.06846
Time:                        16:40:41   Log-Likelihood:                -2617.3
converged:                       True   LL-Null:                       -2809.6
Covariance Type:            nonrobust   LLR p-value:                 6.607e-75
==========================================================================================
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     -5.2320      0.264    -19.813      0.000      -5.750      -4.714
age                        0.0509      0.004     14.014      0.000       0.044       0.058
bmi                        0.0098      0.007      1.467      0.142      -0.003       0.023
smoker                     0.8041      0.092      8.704      0.000       0.623       0.985
diagnosed_afib             2.0335      0.248      8.212      0.000       1.548       2.519
diagnosed_diabetes         0.0338      0.156      0.217      0.829      -0.272       0.339
diagnosed_hypertension    -0.1291      0.121     -1.068      0.285      -0.366       0.108
engagement_level           0.0864      0.189      0.456      0.648      -0.285       0.457
diagnosis_count            0.0450      0.040      1.126      0.260      -0.033       0.123
sleep_quality             -0.1067      0.165     -0.646      0.518      -0.430       0.217
exercise_quality       -8.652e-07   1.38e-06     -0.625      0.532   -3.58e-06    1.85e-06
improving_health          -0.1892      0.547     -0.346      0.729      -1.261       0.883
is_female                  0.1075      0.076      1.419      0.156      -0.041       0.256
==========================================================================================
"""

In [48]:
evaluate_mortality_model(y_val, advanced_model.predict(X_val_const))

{'Log-Loss': 0.2661,
 'Brier Score': 0.07224,
 'C-index (AUC)': 0.68375,
 'Actual Mortality Rate': np.float64(0.081),
 'Predicted Mortality Rate': np.float64(0.08),
 'A/E Ratio': np.float64(1.013)}